## LSTM para previsão de preço de café

# 1. Preparação do ambiente

In [11]:
from IPython.display import display, HTML

display(HTML("<style>.container {width: 100% !important;}</style>"))

## 1.1. Importação de bibliotecas

In [12]:
import yfinance as yf
import torch
import torchvision

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1.2. Checagem de GPU

In [13]:
if torch.cuda.is_available():
    print("__CUDNN VERSION:", torch.backends.cudnn.version())
    print("Device Name:", torch.cuda.get_device_name(0))
    device = 'cuda'
else:
    print("CUDA is not available.")
    device = 'cpu'

print('Device:', device)

__CUDNN VERSION: 91900
Device Name: NVIDIA GeForce RTX 5070
Device: cuda


# # 2. Base de dados

## 2.1. Baixar preços de café e dolar

In [14]:
coffee = yf.download("KC=F", start="2012-01-01", end="2025-12-31", interval="1d")
dollar = yf.download("BRL=X", start="2012-01-01", end="2025-12-31", interval="1d")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [15]:
dollar.head()

Price,Close,High,Low,Open,Volume
Ticker,BRL=X,BRL=X,BRL=X,BRL=X,BRL=X
Date,,,,,
2012-01-02,1.8623,1.8690,1.8535,1.8599,0
2012-01-03,1.8701,1.8697,1.8294,1.8577,0
2012-01-04,1.8345,1.8346,1.8131,1.8346,0
2012-01-05,1.8184,1.8455,1.8197,1.8197,0
2012-01-06,1.8430,1.8538,1.8333,1.8377,0


In [16]:
coffee.head()

Price,Close,High,Low,Open,Volume
Ticker,KC=F,KC=F,KC=F,KC=F,KC=F
Date,,,,,
2012-01-03,227.199997,228.000000,222.399994,226.850006,10692
2012-01-04,226.699997,229.149994,224.399994,226.000000,10663
2012-01-05,219.550003,225.800003,217.550003,225.449997,15059
2012-01-06,221.750000,222.399994,216.600006,219.000000,9980
2012-01-09,221.850006,224.500000,217.600006,220.750000,9768


In [17]:
coffee = coffee[['Close']]
dollar = dollar[['Close']]

In [18]:
df = coffee.join(dollar, how='inner', lsuffix='_coffee', rsuffix='_dolar')
df = df.dropna()

df.to_csv('data/coffee_price.csv')
df

Price            Close        
Ticker            KC=F   BRL=X
Date                          
2012-01-03  227.199997  1.8701
2012-01-04  226.699997  1.8345
2012-01-05  219.550003  1.8184
2012-01-06  221.750000  1.8430
2012-01-09  221.850006  1.8476
...                ...     ...
2025-12-23  346.950012  5.5900
2025-12-24  345.149994  5.5185
2025-12-26  350.250000  5.5195
2025-12-29  352.149994  5.5425
2025-12-30  350.200012  5.5691

[3513 rows x 2 columns]

## 2.2. Explorar base de dados